# Guide 3: Embeddings: giving every token a vector

*How LLMs work* · [Read the guide](https://www.vinaygautam.com/blog/how-llms-work/03-embeddings) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/03-embeddings.ipynb)

This notebook holds every code block from Guide 3, in the guide's order: 10 code cells, 10 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- You already built one — 1 code cell
- Why not stop at one-hot? — 1 code cell
- Measuring closeness — 1 code cell
- Where the numbers come from: learning from neighbors — 3 code cells
- Inside an LLM: the table trains with everything else — 1 code cell
- Looking at embeddings without being fooled — 1 code cell
- What one vector per token cannot do — 2 code cells
- What carries forward — no code

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## You already built one

The code below checks it on the small example and then on a matrix with the shape of Guide 1's $W^{(1)}$, for the letter "t" in slot 20. Slot 0 is the space, so "a" is 1 and "t" is 20. This guide's code is numpy 2.2.6, as pinned in the series `requirements.txt` as of October 2026, in blocks that run in order and share variables.

In [2]:
import numpy as np

small = np.array([[1., 2.], [3., 4.], [5., 6.]])   # (3, 2)
print(np.array([0., 1., 0.]) @ small)

rng = np.random.default_rng(0)
W1 = rng.normal(0, 0.01, (27, 16))             # (27, 16) Guide 1's first layer at its start
print(np.array_equal(np.eye(27)[20] @ W1, W1[20]))

[3. 4.]
True


## Why not stop at one-hot?

One-hot vectors are a legitimate way to feed tokens to a network, but they can't express similarity. The usual way to score how alike two vectors are is the **dot product**: multiply them slot by slot and add up the results. Take the 12 distinct words of the toy text from Guides 1 and 2 as a vocabulary and try it on two of them. For the toy examples in this guide, treat each whole word as a token. Guide 2 showed that real tokens are pieces of words.

In [3]:
text = ("the cat sat on the mat and the dog sat on the log and the "
        "cat saw the dog then the dog ran to the cat and the cat ran on the mat")
words = text.split()
vocab = sorted(set(words))
print(vocab)

one_hot = np.eye(len(vocab))                   # (12, 12) one row per word
print(one_hot[vocab.index("cat")] @ one_hot[vocab.index("dog")])

['and', 'cat', 'dog', 'log', 'mat', 'on', 'ran', 'sat', 'saw', 'the', 'then', 'to']
0.0


## Measuring closeness

Here $u$ and $v$ are any two vectors of the same size, $u \cdot v$ is their dot product, and $\lVert u \rVert$ is the length of $u$: the square root of the sum of its squared values. It's also called the norm, which is the name numpy uses. Try $u = [1, 2, 2]$ and $v = [2, 1, 2]$. The dot product is $2 + 2 + 4 = 8$. Each length is $\sqrt{1 + 4 + 4} = 3$. So the cosine is $8/9$, about 0.889.

In [4]:
def cosine(u, v):
    return u @ v / (np.linalg.norm(u) * np.linalg.norm(v))

u, v = np.array([1., 2., 2.]), np.array([2., 1., 2.])
print(u @ v, np.linalg.norm(u), np.linalg.norm(v), f"{cosine(u, v):.3f}")

8.0 3.0 3.0 0.889


## Where the numbers come from: learning from neighbors

There's no activation function and no bias. Guide 1 warned that two layers with no activation between them collapse into one matrix, and that's true here as well: the network computes `E @ W`, a single 12 × 12 matrix of logits. But that matrix is forced to be the product of a 12 × 4 matrix and a 4 × 12 one, so every word has to be summed up in 4 numbers before any prediction is made. The collapse costs us nothing, because the 4-number summary is what we're after and a more expressive predictor isn't.

In [5]:
ids = np.array([vocab.index(w) for w in words])            # (33,) the text as word IDs
pairs = [(ids[i], ids[j]) for i in range(len(ids))
         for j in range(max(0, i - 2), min(len(ids), i + 3)) if i != j]
inputs = np.array([a for a, b in pairs])                   # (n_pairs,) the word we are at
targets = np.array([b for a, b in pairs])                  # (n_pairs,) one of its neighbors
n_pairs = len(pairs)
print(n_pairs, [(vocab[a], vocab[b]) for a, b in pairs[:4]])

126 [('the', 'cat'), ('the', 'sat'), ('cat', 'the'), ('cat', 'sat')]


Training is Guide 1's loop: a forward pass, the cross-entropy loss, backpropagation, and a gradient-descent step with learning rate $\eta$. The forward pass begins with a lookup, `E[inputs]`, in place of a matrix multiply. Like Guide 1, every step uses all 126 pairs at once, which is a toy convenience. The backward pass is Guide 1's too. `dz` starts as predicted minus correct and is divided by `n_pairs` because the loss is an average over the pairs. Each entry of `W` gets its input times its logit's gradient, added up over the pairs, which is `X.T @ dz`. Each embedding gets the logit gradients carried back through `W`, which is `dz @ W.T`.

In [6]:
rng = np.random.default_rng(0)
E = rng.normal(0, 0.1, (12, 4))    # (12, 4) embedding table
W = rng.normal(0, 0.1, (4, 12))    # (4, 12) output matrix
eta = 0.5

for step in range(3001):
    X = E[inputs]                                          # (n_pairs, 4) one embedding per pair
    z = X @ W                                              # (n_pairs, 12) logits
    z = z - z.max(axis=1, keepdims=True)                   # softmax only sees differences; this stops exp overflowing
    p = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)   # (n_pairs, 12) softmax
    if step in (0, 3000):
        print(step, f"{-np.log(p[np.arange(n_pairs), targets]).mean():.3f}")
    dz = p.copy()
    dz[np.arange(n_pairs), targets] -= 1                         # predicted minus correct
    dz /= n_pairs
    dW = X.T @ dz                                          # (4, 12)
    dX = dz @ W.T                                          # (n_pairs, 4)
    dE = np.zeros_like(E)
    np.add.at(dE, inputs, dX)                              # each pair's gradient goes to its word's row
    E -= eta * dE
    W -= eta * dW

0 2.487


3000 1.713


Now compare the embeddings that training produced.

In [7]:
def sim(a, b):
    return cosine(E[vocab.index(a)], E[vocab.index(b)])

for a, b in [("mat", "log"), ("cat", "dog"), ("sat", "ran"), ("cat", "mat"), ("cat", "the")]:
    print(f"{a}, {b}: {sim(a, b):.2f}")

mat, log: 0.99
cat, dog: 0.95
sat, ran: 0.86
cat, mat: 0.72
cat, the: -0.52


## Inside an LLM: the table trains with everything else

The table's size is the vocabulary size times $d_{\text{model}}$. GPT-2 small's has one row of 768 numbers for each of 50,257 tokens:

In [8]:
token_ids = [464, 3797, 3332, 319, 262]            # "The cat sat on the", from Guide 2
W_E = np.zeros((50257, 768), dtype=np.float32)     # a stand-in with the shape of GPT-2 small's table
X = W_E[token_ids]                                 # (5, 768) = (n, d_model)
print(X.shape, W_E.size, f"{W_E.size / 124_439_808:.1%}")
print(50257 * 12288, f"{50257 * 12288 / 175e9:.2%}")   # the largest GPT-3 model's table

(5, 768) 38597376 31.0%
617558016 0.35%


## Looking at embeddings without being fooled

You can't draw a vector of 4 numbers, let alone 768, so every picture of embeddings squashes them down to 2. A standard way to do it is **principal component analysis (PCA)**, which finds the two directions along which the embeddings vary most and plots each one by its position along those two. The code subtracts the mean first, so directions are measured from the center of the cloud. Variation here means how far the points spread around that center. The singular value decomposition (SVD) then factors the centered table into three pieces. Of those pieces, `U` goes unused. `Vt` holds the directions as rows, sorted by how much the points spread along them, and squaring a value of `s` gives a number proportional to that spread. The first two squares over the total is the share of variation the picture keeps.

In [9]:
centered = E - E.mean(axis=0)                      # (12, 4)
U, s, Vt = np.linalg.svd(centered, full_matrices=False)
print(f"{(s[:2] ** 2).sum() / (s ** 2).sum():.3f}")
Vt *= np.sign(Vt[np.arange(4), np.abs(Vt).argmax(axis=1)])[:, None]   # fix each direction's sign
xy = centered @ Vt[:2].T                           # (12, 2) coordinates for the figure
for i in range(0, 12, 4):
    print("  ".join(f"{w} ({a:.2f}, {b:.2f})" for w, (a, b) in zip(vocab[i:i + 4], xy[i:i + 4])))

0.704
and (2.66, 0.45)  cat (-1.42, 1.57)  dog (-1.35, 2.01)  log (-2.37, 0.26)
mat (-2.37, -0.15)  on (2.36, 1.57)  ran (0.19, -1.36)  sat (-0.34, -1.97)
saw (1.37, -0.87)  the (0.19, 0.65)  then (-0.12, -2.68)  to (1.21, 0.53)


## What one vector per token cannot do

The first limit of that design is context. Here are two sentences run through GPT-2's tokenizer with `tiktoken` 0.14.0, the series pin as of October 2026. As in Guide 2, its first call downloads about 1.5 MB of vocabulary files and caches them.

In [10]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
for sentence in ["He sat on the river bank.", "She got a bank loan."]:
    print(enc.encode(sentence))
print(repr(enc.decode([3331])), enc.encode(" cat"), enc.encode(" Cat"))

[1544, 3332, 319, 262, 7850, 3331, 13]
[3347, 1392, 257, 3331, 8063, 13]
' bank' [3797] [5181]


The second limit is order. Average the toy embeddings of the words in "the cat saw the dog", do the same for "the dog saw the cat", and compare.

In [11]:
def mean_embedding(sentence):
    return E[[vocab.index(w) for w in sentence.split()]].mean(axis=0)   # (4,)

print(np.allclose(mean_embedding("the cat saw the dog"), mean_embedding("the dog saw the cat")))

True


---

← [Guide 2: Tokenization](02-tokenizer.ipynb) · [Guide 4: Attention](04-attention.ipynb) →

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).